<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_09-1_RPG_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 1 | AI RPG 게임 마스터: Tool Calling으로 전투 행동 실행

**실행 환경:** Google Colab (기본 과정은 무료·API 키 불필요)

> **오늘의 핵심:** 모델의 도구 호출 요청(JSON)은 *실행 자체가 아닙니다.* **호스트(Python 코드)**가 함수 이름·인자를 확인하고 함수를 실행하며 결과를 모델에 돌려줍니다.

### 학습 목표
1. 함수 이름·설명·입력 스키마(JSON Schema)를 보고 Tool Calling의 구조를 설명한다.
2. 도구 호출 요청과 Python 함수 실행을 구분한다.
3. 잘못된 인자와 업무 규칙 위반(포션 소진·게임 종료)을 구분해 처리한다.
4. 정상·오류·반복 호출을 자동 검사한다.

**실습 구성:** ① 게임 엔진 → ② 도구 선언 → ③ 모의 Tool Calling → ④ 검증·테스트 → ⑤ (선택) 실제 LLM 연결

*교육용 규칙:* 플레이어 HP 100 / 슬라임 HP 60 / 공격력 20 / 포션 2개(30 회복) / 최대 10턴. 난수 없는 결정적 전투로 학습 결과를 재현합니다.

## 0. 실행 안내

1. Colab에서 **런타임 → 모두 실행**하면 기본 실습이 API 키 없이 끝납니다.
2. 게임 상태가 혼란스러우면 `reset_game()`을 호출하세요.
3. 마지막의 **선택 실습**은 기본값 `RUN_REAL_LLM = False`이므로 자동 실행하지 않습니다. 실제 LLM 사용 시 OpenAI API 키와 이용 요금이 필요합니다.

**중요한 구분:** 아래 모의 응답은 사람이 준비한 JSON이며 **LLM이 생성한 결과가 아닙니다.** 선택 실습에서는 실제 모델의 함수 호출 요청을 받습니다.

> **주석 안내:** 각 Python 실행문 바로 위에 해당 줄의 목적을 설명하는 **한글 주석**을 넣었습니다. `#`으로 시작하는 줄은 실행되지 않습니다. 딕셔너리·목록·함수 호출의 괄호도 흐름에 맞춰 설명했습니다. 기존의 수식·함수·테스트 조건은 변경하지 않았습니다.

In [ ]:
# Python 기본 기능을 불러오고 실행 환경을 확인합니다.
# 표준 라이브러리만 사용하므로 별도 설치가 필요하지 않습니다.
# dataclass로 상태를 만들고 asdict로 상태를 딕셔너리로 변환합니다.
from dataclasses import dataclass, asdict
# Any는 여러 데이터 형식을 허용하는 타입 표시에 사용합니다.
from typing import Any
# JSON 문자열을 Python 객체로 변환하거나 출력하기 위해 불러옵니다.
import json

# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("준비 완료: Python 표준 라이브러리만 사용합니다.")


준비 완료: Python 표준 라이브러리만 사용합니다.


## 1. 게임 상태와 규칙 구현

게임 상태는 **호스트 코드가 보유**합니다. 모델은 상태 변경을 직접 수행하지 않고, 변경을 요청하는 도구 호출만 만듭니다.

- `attack_enemy(target)`: 슬라임 HP를 20 감소시킵니다.
- `heal_player()`: 포션이 있으면 HP를 최대 100까지 30 회복합니다.
- `defend()`: 이번 턴 슬라임 공격의 피해를 10에서 5로 낮춥니다.
- `get_game_status()`: 현재 상태만 읽습니다(턴 소모 없음).
- 각 유효한 행동 후 몬스터가 살아 있으면 반격합니다. **게임 종료 후에는 변경이 금지**됩니다.

In [ ]:
# [이 셀의 목적] 게임 상태 구조, 전투 상수, 내부 보조 함수를 만듭니다.
# 게임 상태를 간단한 데이터 객체로 자동 생성하는 데코레이터입니다.
@dataclass
# 게임의 현재 상태를 묶어 관리하는 자료형을 선언합니다.
class GameState:
    # 현재 게임의 원본 상태: 모델 프롬프트가 아니라 Python 메모리에 저장됩니다.
    # 플레이어의 현재 HP의 기본값을 선언합니다.
    player_hp: int = 100
    # 슬라임의 현재 HP의 기본값을 선언합니다.
    enemy_hp: int = 60
    # 남은 포션 개수의 기본값을 선언합니다.
    potions: int = 2
    # 진행한 턴 수의 기본값을 선언합니다.
    turn: int = 0
    status: str = "playing"  # playing / win / lose / timeout

# 전투 규칙은 한 곳에 모아 필요할 때 바꾸기 쉽게 만듭니다.
# 플레이어의 최대 체력을 설정합니다.
PLAYER_MAX_HP = 100
# 한 번 공격할 때 슬라임이 받는 피해량을 정합니다.
PLAYER_DAMAGE = 20
# 반격 시 플레이어가 받는 기본 피해량을 정합니다.
ENEMY_DAMAGE = 10
# 포션 한 개로 회복할 체력 수치를 정합니다.
POTION_HEAL = 30
# 게임이 무한 반복하지 않도록 최대 턴을 설정합니다.
MAX_TURNS = 10

# 초깃값으로 게임 상태를 생성하거나 초기화합니다.
game = GameState()
# 도구 요청과 응답을 남길 빈 로그 리스트를 만듭니다.
call_log: list[dict[str, Any]] = []

# 게임과 호출 기록을 처음 상태로 되돌리는 함수를 정의합니다.
def reset_game() -> dict[str, Any]:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """게임 상태와 호출 기록을 모두 초기화합니다."""
    # 함수 바깥에 있는 게임 상태 변수를 새 객체로 교체하도록 지정합니다.
    global game
    # 초깃값으로 게임 상태를 생성하거나 초기화합니다.
    game = GameState()
    # 이전 게임에서 남은 도구 호출 기록을 지웁니다.
    call_log.clear()
    # 데이터 클래스를 Python 딕셔너리로 바꿔 반환합니다.
    return asdict(game)

# 실패 원인과 복구 힌트를 담는 오류 응답 함수를 정의합니다.
def _error(code: str, message: str, **extra: Any) -> dict[str, Any]:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """모델이 다음 행동을 결정하기 쉽도록 오류를 구조화합니다."""
    # 필요한 항목을 딕셔너리로 묶어 반환합니다.
    return {"ok": False, "error_code": code, "message": message, **extra}

# 슬라임의 반격과 방어 효과를 처리하는 내부 함수를 정의합니다.
def _enemy_counterattack(defending: bool = False) -> None:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """유효한 행동 후 살아 있는 슬라임이 1회 반격합니다."""
    # 몬스터가 살아 있을 때에만 반격하도록 합니다.
    if game.enemy_hp > 0:
        # 방어 상태면 반격 피해를 절반으로 계산합니다.
        damage = ENEMY_DAMAGE // 2 if defending else ENEMY_DAMAGE
        # 체력 감소·회복 결과가 허용 범위를 벗어나지 않게 합니다.
        game.player_hp = max(0, game.player_hp - damage)

# 한 턴을 증가시키고 승패·시간 초과를 판정하는 함수를 정의합니다.
def _finish_turn() -> None:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """한 턴 종료 뒤 승리/패배/턴 제한을 판정합니다."""
    # 행동을 마친 뒤 게임 턴 수를 하나 증가시킵니다.
    game.turn += 1
    # 몬스터 HP가 0 이하이면 플레이어 승리입니다.
    if game.enemy_hp <= 0:
        # 판정된 게임 종료 상태를 기록합니다.
        game.status = "win"
    # 플레이어 HP가 0 이하일 때 패배를 표시합니다.
    elif game.player_hp <= 0:
        # 판정된 게임 종료 상태를 기록합니다.
        game.status = "lose"
    # 정해진 최대 턴을 넘으면 시간 초과로 끝냅니다.
    elif game.turn >= MAX_TURNS:
        # 판정된 게임 종료 상태를 기록합니다.
        game.status = "timeout"

# 현재 게임 상태를 딕셔너리 응답으로 포장하는 함수를 정의합니다.
def _snapshot(action: str) -> dict[str, Any]:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """성공 응답에는 항상 최신 게임 상태를 포함합니다."""
    # 필요한 항목을 딕셔너리로 묶어 반환합니다.
    return {"ok": True, "action": action, "state": asdict(game)}

# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print(reset_game())


{'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'status': 'playing'}


### 1-1. 실제로 실행할 게임 함수

다음 함수들은 **도구 실행 측(Host)** 코드입니다. 프롬프트나 JSON 자체로는 실행되지 않습니다.

In [ ]:
# [이 셀의 목적] 실제로 게임 상태를 변경하는 함수 네 개를 정의합니다.
# 게임을 변경하지 않는 조회 함수를 정의합니다.
def get_game_status() -> dict[str, Any]:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """플레이어·몬스터 HP와 포션, 턴 수를 조회합니다. 상태는 변경하지 않습니다."""
    # 실행이 끝난 뒤 현재 게임 상태를 응답으로 반환합니다.
    return _snapshot("get_game_status")

# 지정된 슬라임에 대한 공격 함수를 정의합니다.
def attack_enemy(target: str) -> dict[str, Any]:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """target='slime'일 때만 공격합니다. 유효한 공격은 턴을 소모합니다."""
    # 게임이 이미 종료되었다면 현재 행동을 거부합니다.
    if game.status != "playing":
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("GAME_OVER", "종료된 게임에서는 공격할 수 없습니다.")
    # 지원하지 않는 공격 대상인지 검사합니다.
    if target != "slime":
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("UNKNOWN_TARGET", "공격 대상이 없습니다.", allowed_targets=["slime"])

    # 공격 피해를 적용하되 HP가 음수가 되지 않도록 합니다.
    game.enemy_hp = max(0, game.enemy_hp - PLAYER_DAMAGE)
    _enemy_counterattack()  # 슬라임이 죽었다면 이 함수는 아무 일도 하지 않습니다.
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    _finish_turn()
    # 실행이 끝난 뒤 현재 게임 상태를 응답으로 반환합니다.
    return _snapshot("attack_enemy")

# 포션을 사용해 체력을 회복하는 함수를 정의합니다.
def heal_player() -> dict[str, Any]:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """포션 1개를 쓰고 체력을 회복합니다. 포션이 없으면 턴을 소모하지 않습니다."""
    # 게임이 이미 종료되었다면 현재 행동을 거부합니다.
    if game.status != "playing":
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("GAME_OVER", "종료된 게임에서는 회복할 수 없습니다.")
    # 남은 포션이 없으면 회복할 수 없습니다.
    if game.potions == 0:
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("NO_POTION", "포션이 없습니다. 공격이나 방어를 선택하세요.", available_actions=["attack_enemy", "defend", "get_game_status"])
    # HP가 이미 가득 찼으면 포션을 낭비하지 않습니다.
    if game.player_hp == PLAYER_MAX_HP:
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("HP_FULL", "HP가 최대치이므로 포션을 소모하지 않습니다.")

    # 성공한 회복 행동에서 포션 한 개를 차감합니다.
    game.potions -= 1
    # 체력 감소·회복 결과가 허용 범위를 벗어나지 않게 합니다.
    game.player_hp = min(PLAYER_MAX_HP, game.player_hp + POTION_HEAL)
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    _enemy_counterattack()
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    _finish_turn()
    # 실행이 끝난 뒤 현재 게임 상태를 응답으로 반환합니다.
    return _snapshot("heal_player")

# 적의 반격 피해를 감소시키는 방어 함수를 정의합니다.
def defend() -> dict[str, Any]:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """방어 1턴 동안 몬스터 반격 피해를 절반으로 줄입니다."""
    # 게임이 이미 종료되었다면 현재 행동을 거부합니다.
    if game.status != "playing":
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("GAME_OVER", "종료된 게임에서는 방어할 수 없습니다.")
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    _enemy_counterattack(defending=True)
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    _finish_turn()
    # 실행이 끝난 뒤 현재 게임 상태를 응답으로 반환합니다.
    return _snapshot("defend")

# 사람이 함수를 직접 호출하면 게임 상태가 즉시 변경됩니다.
# 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
reset_game()
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("상태 조회:", get_game_status())
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("공격 실행:", attack_enemy("slime"))


상태 조회: {'ok': True, 'action': 'get_game_status', 'state': {'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'status': 'playing'}}
공격 실행: {'ok': True, 'action': 'attack_enemy', 'state': {'player_hp': 90, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'status': 'playing'}}


## 2. 모델에 보여줄 도구 정의(JSON Schema)

모델에게 전달되는 것은 **함수 이름, 설명, 파라미터 스키마**입니다. 내부 Python 구현이나 게임 변수 자체가 모델에 자동으로 전달되지는 않습니다.

`strict: true`는 지원되는 JSON Schema 형식을 준수하도록 돕지만 **포션 수·권한·종료 여부 같은 업무 규칙 검증을 대체하지 않습니다.**

In [ ]:
# [이 셀의 목적] 모델에 공개할 도구의 이름·설명·입력 형식을 작성합니다.
# Responses API의 function tool 정의 형태를 사용합니다.
# 모든 객체 스키마는 additionalProperties=False로 제한합니다.
# 인자가 없는 함수는 빈 properties와 required=[]를 사용합니다.
# 입력값이 없는 도구에 사용할 JSON Schema를 만듭니다.
def no_args_schema() -> dict[str, Any]:
    # 필요한 항목을 딕셔너리로 묶어 반환합니다.
    return {"type": "object", "properties": {}, "required": [], "additionalProperties": False}

# 모델에 전달할 네 가지 도구의 사양 목록을 시작합니다.
TOOLS = [
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    {
        # 모델에게 이 항목이 함수 호출 도구임을 알립니다.
        "type": "function", "name": "attack_enemy",
        # 모델의 도구 선택에 도움이 되는 기능 설명을 정의합니다.
        "description": "슬라임을 공격합니다. 대상은 slime만 허용하며, 공격에 성공하면 한 턴 진행합니다.",
        # 모델이 생성할 인자의 JSON Schema를 정의합니다.
        "parameters": {
            # 인자는 JSON 객체이며 지정된 target 필드만 받습니다.
            "type": "object", "properties": {"target": {"type": "string", "enum": ["slime"], "description": "공격 대상"}},
            # 필수 인자와 추가 인자 허용 여부를 지정합니다.
            "required": ["target"], "additionalProperties": False,
        # 스키마 준수를 요청하는 strict 설정을 켭니다.
        }, "strict": True,
    # 앞에서 시작한 딕셔너리 항목을 닫습니다.
    },
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    {
        # 모델에게 이 항목이 함수 호출 도구임을 알립니다.
        "type": "function", "name": "heal_player",
        # 모델의 도구 선택에 도움이 되는 기능 설명을 정의합니다.
        "description": "포션으로 플레이어 HP를 회복합니다. 포션과 HP 상태를 Python 코드가 검증합니다.",
        # 모델이 생성할 인자의 JSON Schema를 정의합니다.
        "parameters": no_args_schema(), "strict": True,
    # 앞에서 시작한 딕셔너리 항목을 닫습니다.
    },
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    {
        # 모델에게 이 항목이 함수 호출 도구임을 알립니다.
        "type": "function", "name": "defend",
        # 모델의 도구 선택에 도움이 되는 기능 설명을 정의합니다.
        "description": "한 턴 방어해 몬스터 반격 피해를 줄입니다.",
        # 모델이 생성할 인자의 JSON Schema를 정의합니다.
        "parameters": no_args_schema(), "strict": True,
    # 앞에서 시작한 딕셔너리 항목을 닫습니다.
    },
    # 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
    {
        # 모델에게 이 항목이 함수 호출 도구임을 알립니다.
        "type": "function", "name": "get_game_status",
        # 모델의 도구 선택에 도움이 되는 기능 설명을 정의합니다.
        "description": "현재 게임 상태(HP, 포션, 턴, 종료 상태)를 조회합니다. 턴을 소모하지 않습니다.",
        # 모델이 생성할 인자의 JSON Schema를 정의합니다.
        "parameters": no_args_schema(), "strict": True,
    # 앞에서 시작한 딕셔너리 항목을 닫습니다.
    },
# 앞에서 시작한 목록 정의를 마칩니다.
]

# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print(json.dumps(TOOLS, ensure_ascii=False, indent=2))


[
  {
    "type": "function",
    "name": "attack_enemy",
    "description": "슬라임을 공격합니다. 대상은 slime만 허용하며, 공격에 성공하면 한 턴 진행합니다.",
    "parameters": {
      "type": "object",
      "properties": {
        "target": {
          "type": "string",
          "enum": [
            "slime"
          ],
          "description": "공격 대상"
        }
      },
      "required": [
        "target"
      ],
      "additionalProperties": false
    },
    "strict": true
  },
  {
    "type": "function",
    "name": "heal_player",
    "description": "포션으로 플레이어 HP를 회복합니다. 포션과 HP 상태를 Python 코드가 검증합니다.",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": [],
      "additionalProperties": false
    },
    "strict": true
  },
  {
    "type": "function",
    "name": "defend",
    "description": "한 턴 방어해 몬스터 반격 피해를 줄입니다.",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": [],
      "additionalProperties": false
    },
    "strict": true
  },
 

## 3. Host의 검증 및 도구 실행기 만들기

중요: JSON의 `name`을 보고 **미리 허용한 함수만** 실행합니다. `eval()`이나 임의의 Python 코드 실행은 사용하지 않습니다.

실제 모델이 만든 인자도 사용자 입력처럼 검사해야 합니다. 여기서는 표준 라이브러리만으로 학습에 필요한 타입·필수값·허용값 검사를 구현합니다.

In [ ]:
# [이 셀의 목적] 도구 호출을 허용 목록과 스키마로 검증한 뒤 실행합니다.
# 실행 가능한 함수는 명시적으로 허용 목록에 등록합니다.
# 허용한 Python 함수만 도구 이름과 연결합니다.
ALLOWED_FUNCTIONS = {
    # 도구 이름 또는 요청 내용을 문자열로 설정합니다.
    "attack_enemy": attack_enemy,
    # 도구 이름 또는 요청 내용을 문자열로 설정합니다.
    "heal_player": heal_player,
    # 도구 이름 또는 요청 내용을 문자열로 설정합니다.
    "defend": defend,
    # 도구 이름 또는 요청 내용을 문자열로 설정합니다.
    "get_game_status": get_game_status,
# 앞에서 시작한 딕셔너리 항목을 닫습니다.
}
# 도구 이름으로 JSON Schema를 빨리 찾도록 사전을 만듭니다.
TOOL_DEFINITIONS = {tool["name"]: tool for tool in TOOLS}

# 도구 이름과 인자가 유효한지 사전 검사하는 함수를 정의합니다.
def validate_call(name: str, args: Any) -> dict[str, Any] | None:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """오류가 없으면 None, 문제가 있으면 구조화된 오류를 반환합니다."""
    # 사전에 등록하지 않은 도구는 실행하지 않습니다.
    if name not in ALLOWED_FUNCTIONS:
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("UNKNOWN_TOOL", "허용하지 않는 도구입니다.", allowed_tools=list(ALLOWED_FUNCTIONS))
    # 도구 인자가 딕셔너리인지 타입을 검사합니다.
    if not isinstance(args, dict):
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("INVALID_ARGUMENTS", "arguments는 JSON 객체여야 합니다.")

    # 현재 도구의 입력 스키마를 가져옵니다.
    schema = TOOL_DEFINITIONS[name]["parameters"]
    # 반드시 있어야 하는 필수 인자 목록을 집합으로 바꿉니다.
    required = set(schema["required"])
    # 실제로 전달된 인자 이름을 집합으로 바꿉니다.
    provided = set(args)
    # 필수 인자가 빠진 경우 오류를 반환합니다.
    if required - provided:
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("MISSING_ARGUMENT", "필수 인자가 빠졌습니다.", missing=sorted(required - provided))
    # 스키마에 없는 추가 인자가 있는지 검사합니다.
    if provided - set(schema["properties"]):
        # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
        return _error("EXTRA_ARGUMENT", "정의되지 않은 인자를 전달했습니다.", extra=sorted(provided - set(schema["properties"])))
    # 전달된 인자를 하나씩 검사합니다.
    for key, value in args.items():
        # 현재 인자에 적용되는 타입·허용값 규칙을 확인합니다.
        prop = schema["properties"][key]
        # 문자열로 지정된 인자에 잘못된 타입이 들어왔는지 검사합니다.
        if prop["type"] == "string" and not isinstance(value, str):
            # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
            return _error("INVALID_TYPE", f"{key}에는 문자열이 필요합니다.")
        # 인자 값이 허용된 선택지 중 하나인지 검사합니다.
        if "enum" in prop and value not in prop["enum"]:
            # 사용자와 모델이 원인을 파악할 수 있는 오류 응답을 반환합니다.
            return _error("INVALID_VALUE", f"{key}에 허용되지 않은 값입니다.", allowed_values=prop["enum"])
    # 검증 오류가 없음을 뜻하는 None을 반환합니다.
    return None

# JSON 요청을 해석하고 검증 후 등록된 Python 함수를 실행하는 함수를 정의합니다.
def execute_tool_call(name: str, arguments: str | dict[str, Any]) -> dict[str, Any]:
    # 함수 목적과 반환값을 설명하는 문서 문자열입니다.
    """모델의 요청을 검증한 뒤 허용된 함수만 실행합니다."""
    # 실패할 수 있는 구문을 안전하게 실행합니다.
    try:
        # JSON 문자열이면 딕셔너리로 변환하고 아니면 원본을 사용합니다.
        args = json.loads(arguments) if isinstance(arguments, str) else arguments
    # JSON 문자열이 잘못된 경우 예외를 잡아 안전하게 처리합니다.
    except json.JSONDecodeError:
        # 호출 또는 계산 결과를 변수에 저장합니다.
        result = _error("INVALID_JSON", "arguments를 JSON으로 해석할 수 없습니다.")
    # 앞선 조건이 성립하지 않은 경우를 처리합니다.
    else:
        # 실행 전 도구 이름과 인자를 검사합니다.
        problem = validate_call(name, args)
        # 사전 검증에서 오류가 발견되었는지 확인합니다.
        if problem is not None:
            # 호출 또는 계산 결과를 변수에 저장합니다.
            result = problem
        # 앞선 조건이 성립하지 않은 경우를 처리합니다.
        else:
            # 실제 실행 지점: 모델이 아니라 내 Python 함수가 실행됩니다.
            # 실패할 수 있는 구문을 안전하게 실행합니다.
            try:
                # 허용된 함수만 이름으로 찾아 검증된 인자를 전달해 실행합니다.
                result = ALLOWED_FUNCTIONS[name](**args)
            # 예상하지 못한 내부 실행 오류를 외부에 상세 노출하지 않습니다.
            except Exception:
                # 교육용: 내부 예외의 세부 정보는 모델에 그대로 노출하지 않습니다.
                # 호출 또는 계산 결과를 변수에 저장합니다.
                result = _error("INTERNAL_ERROR", "도구 실행 중 내부 오류가 발생했습니다.")
    # 나중에 검토할 수 있도록 호출 입력과 실행 결과를 기록합니다.
    call_log.append({"tool": name, "arguments": arguments, "result": result})
    # 최종 도구 실행 결과를 호출자에게 반환합니다.
    return result

# 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
reset_game()
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print(execute_tool_call("attack_enemy", '{"target":"slime"}'))
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("호출 로그:", call_log[-1])


{'ok': True, 'action': 'attack_enemy', 'state': {'player_hp': 90, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'status': 'playing'}}
호출 로그: {'tool': 'attack_enemy', 'arguments': '{"target":"slime"}', 'result': {'ok': True, 'action': 'attack_enemy', 'state': {'player_hp': 90, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'status': 'playing'}}}


## 4. 모의 Tool Calling: 요청 → 검증 → 실행 → 결과

모의 호출을 활용하면 **호출 요청과 실제 실행이 다른 단계**라는 사실을 API 키 없이 재현할 수 있습니다. 아래 `mock_model_output`은 모형 데이터로서, 모델의 선택 능력을 평가하는 결과가 아닙니다.

In [ ]:
# [이 셀의 목적] 미리 작성한 모델 출력 예시로 도구 요청과 실행을 비교합니다.
# 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
reset_game()
# 실제 LLM 없이 연습할 모의 함수 호출 응답을 만듭니다.
mock_model_output = {
    # 도구 이름 또는 요청 내용을 문자열로 설정합니다.
    "type": "function_call",      # 실제 Responses API에서도 쓰이는 출력 항목 유형
    # 도구 요청과 도구 실행 결과를 연결할 식별자입니다.
    "call_id": "call_demo_01",     # 호출 결과와 요청을 연결하는 식별자
    # 실행할 도구의 이름을 나타냅니다.
    "name": "attack_enemy",        # 모델이 요청했다고 가정한 도구 이름
    # 도구에 전달할 인자를 JSON 문자열로 명시합니다.
    "arguments": '{"target":"slime"}',  # 인자는 JSON 문자열
# 앞에서 시작한 딕셔너리 항목을 닫습니다.
}

# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("1) 요청만 생성됨. HP는 아직 변하지 않았습니다:", game.enemy_hp)
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("2) 요청 JSON:", mock_model_output)
# 모의 도구 요청을 실행하고 결과를 변수에 저장합니다.
result = execute_tool_call(mock_model_output["name"], mock_model_output["arguments"])
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("3) Host 실행 결과:", json.dumps(result, ensure_ascii=False))
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("4) 실행 후 슬라임 HP:", game.enemy_hp)
# 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
assert game.enemy_hp == 40


1) 요청만 생성됨. HP는 아직 변하지 않았습니다: 60
2) 요청 JSON: {'type': 'function_call', 'call_id': 'call_demo_01', 'name': 'attack_enemy', 'arguments': '{"target":"slime"}'}
3) Host 실행 결과: {"ok": true, "action": "attack_enemy", "state": {"player_hp": 90, "enemy_hp": 40, "potions": 2, "turn": 1, "status": "playing"}}
4) 실행 후 슬라임 HP: 40


### 4-1. 여러 번 호출하는 경우

`get_game_status`는 읽기 전용이며, `attack_enemy`는 상태를 변경합니다. 반복 공격 후 몬스터가 쓰러지면 **추가 공격이 거부**됩니다.

In [ ]:
# [이 셀의 목적] 여러 호출을 순서대로 수행하고 게임 종료 후 호출을 검사합니다.
# 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
reset_game()
# 연속 호출 및 게임 종료 테스트에 사용할 행동 순서를 정합니다.
scenario = [
    # 반복 실행할 함수 이름과 인자를 한 항목으로 묶습니다.
    ("get_game_status", {}),
    # 반복 실행할 함수 이름과 인자를 한 항목으로 묶습니다.
    ("attack_enemy", {"target": "slime"}),
    # 반복 실행할 함수 이름과 인자를 한 항목으로 묶습니다.
    ("attack_enemy", {"target": "slime"}),
    # 반복 실행할 함수 이름과 인자를 한 항목으로 묶습니다.
    ("attack_enemy", {"target": "slime"}),
    ("attack_enemy", {"target": "slime"}),  # 이미 이겼다면 GAME_OVER
# 앞에서 시작한 목록 정의를 마칩니다.
]
# 여러 도구 호출 시나리오를 순서대로 실행합니다.
for name, args in scenario:
    # 모의 도구 요청을 실행하고 결과를 변수에 저장합니다.
    result = execute_tool_call(name, args)
    # 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
    print(f"{name:16s} → {result.get('error_code', 'OK'):10s} | 상태: {asdict(game)}")
# 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
assert game.status == "win"
# 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
assert call_log[-1]["result"]["error_code"] == "GAME_OVER"


get_game_status  → OK         | 상태: {'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'status': 'playing'}
attack_enemy     → OK         | 상태: {'player_hp': 90, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'status': 'playing'}
attack_enemy     → OK         | 상태: {'player_hp': 80, 'enemy_hp': 20, 'potions': 2, 'turn': 2, 'status': 'playing'}
attack_enemy     → OK         | 상태: {'player_hp': 80, 'enemy_hp': 0, 'potions': 2, 'turn': 3, 'status': 'win'}
attack_enemy     → GAME_OVER  | 상태: {'player_hp': 80, 'enemy_hp': 0, 'potions': 2, 'turn': 3, 'status': 'win'}


## 5. 오류 테스트: 스키마 오류 vs 업무 규칙 오류

- **스키마 오류:** 필수 인자 누락, 잘못된 타입, 미허용 도구 → 함수 실행 전에 검증기에서 거부합니다.
- **업무 규칙 오류:** 포션 없음, 이미 종료된 게임 → 실행 함수가 현재 상태를 보고 거부합니다.
- `strict: true`를 쓰더라도 **업무 규칙 검증은 반드시 코드에서** 수행합니다.

In [ ]:
# [이 셀의 목적] 잘못된 도구 요청 여섯 가지가 거부되는지 확인합니다.
# 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
reset_game()
# 검증기에 전달할 잘못된 요청 목록을 만듭니다.
invalid_cases = [
    ("attack_enemy", {}),                          # 필수 인자 누락
    ("attack_enemy", {"target": 123}),             # 타입 오류
    ("attack_enemy", {"target": "dragon"}),       # 허용값 위반
    ("attack_enemy", {"target": "slime", "hp": 0}), # 정의하지 않은 인자
    ("delete_all_files", {}),                       # 허용 목록에 없는 도구
    ("attack_enemy", '{broken-json'),               # JSON 형식 오류
# 앞에서 시작한 목록 정의를 마칩니다.
]
# 여러 도구 호출 시나리오를 순서대로 실행합니다.
for name, args in invalid_cases:
    # 잘못된 도구 요청을 실행하여 오류 응답을 확인합니다.
    response = execute_tool_call(name, args)
    # 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
    print(f"{name:16s} → {response['error_code']}: {response['message']}")
    # 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
    assert response["ok"] is False

# 잘못된 호출은 게임 상태를 바꾸지 않아야 합니다.
# 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
assert asdict(game) == asdict(GameState())


attack_enemy     → MISSING_ARGUMENT: 필수 인자가 빠졌습니다.
attack_enemy     → INVALID_TYPE: target에는 문자열이 필요합니다.
attack_enemy     → INVALID_VALUE: target에 허용되지 않은 값입니다.
attack_enemy     → EXTRA_ARGUMENT: 정의되지 않은 인자를 전달했습니다.
delete_all_files → UNKNOWN_TOOL: 허용하지 않는 도구입니다.
attack_enemy     → INVALID_JSON: arguments를 JSON으로 해석할 수 없습니다.


In [ ]:
# [이 셀의 목적] 회복 포션 관련 업무 규칙과 구조화 오류를 확인합니다.
# 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
reset_game()
# 포션은 2개지만 처음 HP 100은 최대치이므로 바로 회복할 수 없습니다.
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("HP가 가득 찬 경우:", execute_tool_call("heal_player", {}))
# 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
assert game.potions == 2

# 공격 후 반격을 받아 HP가 감소하면 포션을 사용할 수 있습니다.
# 도구 이름과 인자를 실행기에 전달합니다.
execute_tool_call("attack_enemy", {"target": "slime"})
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("회복 1:", execute_tool_call("heal_player", {}))
# 도구 이름과 인자를 실행기에 전달합니다.
execute_tool_call("attack_enemy", {"target": "slime"})
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("회복 2:", execute_tool_call("heal_player", {}))
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("포션을 모두 쓴 후:", execute_tool_call("heal_player", {}))
# 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
assert game.potions == 0
# 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
assert call_log[-1]["result"]["error_code"] == "NO_POTION"

# 오류 응답에는 '다음에 선택할 수 있는 행동'도 포함됩니다.
# 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
print("선택 가능한 행동:", call_log[-1]["result"]["available_actions"])


HP가 가득 찬 경우: {'ok': False, 'error_code': 'HP_FULL', 'message': 'HP가 최대치이므로 포션을 소모하지 않습니다.'}
회복 1: {'ok': True, 'action': 'heal_player', 'state': {'player_hp': 90, 'enemy_hp': 40, 'potions': 1, 'turn': 2, 'status': 'playing'}}
회복 2: {'ok': True, 'action': 'heal_player', 'state': {'player_hp': 90, 'enemy_hp': 20, 'potions': 0, 'turn': 4, 'status': 'playing'}}
포션을 모두 쓴 후: {'ok': False, 'error_code': 'NO_POTION', 'message': '포션이 없습니다. 공격이나 방어를 선택하세요.', 'available_actions': ['attack_enemy', 'defend', 'get_game_status']}
선택 가능한 행동: ['attack_enemy', 'defend', 'get_game_status']


## 6. 자동 검증 (Contract Test)

아래 테스트는 초깃값, 상태 조회의 읽기 전용 특성, 공격 동작, 잘못된 인자 거부, 포션 오류, 게임 종료 후 거부를 확인합니다. **여기까지 모두 실행되면 기본 실습을 완료한 것입니다.**

In [ ]:
# [이 셀의 목적] 6개의 자동 테스트로 게임과 도구 계약을 검증합니다.
# 정상·오류 상황에 대한 자동 검증 함수를 정의합니다.
def run_contract_tests() -> None:
    # 1) 초기 상태가 정확한지 확인
    # 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
    reset_game()
    # 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
    assert (game.player_hp, game.enemy_hp, game.potions, game.turn) == (100, 60, 2, 0)

    # 2) 조회 도구는 턴을 소모하지 않아야 함
    # 도구 이름과 인자를 실행기에 전달합니다.
    execute_tool_call("get_game_status", {})
    # 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
    assert game.turn == 0

    # 3) 유효한 공격은 HP를 변경하고 턴을 진행함
    # 정상 공격 결과를 저장하여 HP 변경을 검증합니다.
    attack_result = execute_tool_call("attack_enemy", {"target": "slime"})
    # 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
    assert attack_result["ok"] is True and game.enemy_hp == 40 and game.turn == 1

    # 4) 잘못된 스키마는 상태를 바꾸지 않음
    # 오류 호출이 상태를 바꾸지 않았는지 비교할 상태 복사본을 만듭니다.
    previous = asdict(game).copy()
    # 잘못된 타입의 공격 요청을 실행해 오류를 확인합니다.
    invalid = execute_tool_call("attack_enemy", {"target": 123})
    # 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
    assert invalid["error_code"] == "INVALID_TYPE" and asdict(game) == previous

    # 5) 포션 소진 시 NO_POTION 반환
    # 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
    reset_game()
    game.player_hp = 40  # 테스트용 상태 준비 (실제 게임 동작이 아님)
    # 도구 이름과 인자를 실행기에 전달합니다.
    execute_tool_call("heal_player", {})
    # 도구 이름과 인자를 실행기에 전달합니다.
    execute_tool_call("heal_player", {})
    # 포션 소진 상황의 오류 응답을 저장합니다.
    no_potion = execute_tool_call("heal_player", {})
    # 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
    assert no_potion["error_code"] == "NO_POTION" and game.potions == 0

    # 6) 종료된 게임은 HP가 더 바뀌지 않음
    # 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
    reset_game()
    # 반복 공격으로 몬스터를 쓰러뜨리는 상황을 만듭니다.
    for _ in range(3):
        # 도구 이름과 인자를 실행기에 전달합니다.
        execute_tool_call("attack_enemy", {"target": "slime"})
    # 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
    assert game.status == "win"
    # 오류 호출이 상태를 바꾸지 않았는지 비교할 상태 복사본을 만듭니다.
    previous = asdict(game).copy()
    # 승리 후 공격이 거부되는지 확인합니다.
    after_win = execute_tool_call("attack_enemy", {"target": "slime"})
    # 조건이 맞지 않으면 AssertionError를 내어 실습 검증을 실패 처리합니다.
    assert after_win["error_code"] == "GAME_OVER" and asdict(game) == previous

    # 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
    print("✅ Contract Test 6개 모두 통과")

# 앞 단계에서 준비한 데이터와 설정을 사용해 다음 처리를 진행합니다.
run_contract_tests()


✅ Contract Test 6개 모두 통과


## 7. (선택) 실제 LLM을 연결해 Tool Calling 확인하기

이전 단계까지는 사람이 준비한 JSON으로 실행 흐름을 학습했습니다. 이제 **실제 모델이 네 가지 도구 중 무엇을 호출할지 선택**하도록 해 봅니다.

- 별도의 OpenAI API 키가 필요하며 **호출 비용이 발생할 수 있습니다.**
- 키는 코드에 직접 저장하지 않고 실행할 때만 입력합니다.
- 실수로 무한 호출하지 않도록 **호출 횟수를 4회로 제한**합니다.
- 새 전투가 시작되도록 상태를 초기화합니다.
- 각 함수 호출은 모델이 아니라 `execute_tool_call()`에서 실제 실행됩니다.

참고: [OpenAI 공식 Function Calling 문서](https://developers.openai.com/api/docs/guides/function-calling)

In [ ]:
# [이 셀의 목적] 선택적으로 실제 OpenAI API에 연결해 도구 호출 루프를 확인합니다.
# 다음 코드는 선택 실행입니다. 전체 실행 시 외부 API를 호출하지 않습니다.
# 실제 API 호출을 실행할지 설정합니다. 기본값 False는 비용이 들지 않습니다.
RUN_REAL_LLM = False
MODEL_NAME = "gpt-5.4-mini"  # 사용 가능한 모델로 바꾸어도 됩니다. 계정별 접근 권한과 요금 확인.

# 기본 실행에서는 외부 API에 접속하지 않도록 분기합니다.
if not RUN_REAL_LLM:
    # 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
    print("선택 실습은 건너뜁니다. 실제 실행하려면 RUN_REAL_LLM=True로 바꾸세요.")
# 앞선 조건이 성립하지 않은 경우를 처리합니다.
else:
    # Colab 환경에 OpenAI SDK 설치 (선택 실습에서만 수행)
    # 선택 실습에서 pip 설치 명령을 실행할 모듈을 불러옵니다.
    import subprocess, sys
    # 실제 LLM 실습 때만 OpenAI SDK를 설치합니다.
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-U", "openai"])

    # 화면에 노출하지 않고 API 키를 입력받기 위해 불러옵니다.
    from getpass import getpass
    # OpenAI API를 호출할 클라이언트 클래스를 불러옵니다.
    from openai import OpenAI
    # 입력된 글자가 화면에 보이지 않게 API 키를 받습니다.
    api_key = getpass("OpenAI API 키(입력 내용 숨김): ")
    # API 키를 입력하지 않았다면 실제 모델 실습을 중단합니다.
    if not api_key:
        # API 키가 없는 경우 명확한 오류를 발생시킵니다.
        raise ValueError("실제 LLM 실습에는 API 키가 필요합니다.")
    # 입력받은 API 키로 API 호출용 클라이언트를 생성합니다.
    llm_client = OpenAI(api_key=api_key)
    # 검증에 앞서 현재 게임을 초기 상태로 되돌립니다.
    reset_game()

    # 도구 호출을 포함해 상태를 요약해 답하게 유도합니다.
    # 모델에게 전달할 게임 진행 요청을 작성합니다.
    prompt = "현재 게임 상태를 확인한 뒤 슬라임을 한 번 공격하고 결과를 한국어로 설명해줘."
    # 모델의 출력 또는 다음 도구 호출 요청을 받습니다.
    response = llm_client.responses.create(
        # 사용할 LLM 모델을 지정합니다.
        model=MODEL_NAME,
        # 모델이 따라야 할 역할 및 도구 사용 원칙을 지정합니다.
        instructions="너는 텍스트 RPG 도우미다. 상태는 도구 결과로만 확인한다. 필요한 도구만 호출하라.",
        # 모델에게 사용자 요청이나 실행 결과를 전달합니다.
        input=prompt,
        # 모델이 선택할 수 있는 도구 목록을 전달합니다.
        tools=TOOLS,
        parallel_tool_calls=False,  # 교육용으로 한 번에 최대 한 도구 요청
    # 앞에서 시작한 함수 호출 또는 표현식을 닫습니다.
    )

    # 모델은 호출 요청을 반환할 뿐, 게임 함수를 직접 실행하지 못합니다.
    # 최대 네 번까지 모델의 도구 호출과 결과 전달을 반복합니다.
    for step in range(4):
        # 모델 응답 중 함수 호출 요청만 골라냅니다.
        calls = [item for item in response.output if item.type == "function_call"]
        # 이번 응답에 도구 호출이 없으면 최종 답변으로 처리합니다.
        if not calls:
            # 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
            print("모델 최종 답변:", response.output_text)
            # 모델의 최종 답변이 나와 반복을 종료합니다.
            break
        # 모델에 다시 전달할 도구 실행 결과 목록을 만듭니다.
        outputs = []
        # 모델이 요청한 각각의 도구를 처리합니다.
        for call in calls:
            # 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
            print("모델 호출 요청:", call.name, call.arguments)
            # 모델의 요청을 Host 실행기에 넘겨 실제 Python 함수를 실행합니다.
            tool_result = execute_tool_call(call.name, call.arguments)
            # 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
            print("호스트 실행 결과:", tool_result)
            # call_id가 모델이 만든 호출과 실행 결과를 연결해 줍니다.
            # 호출 ID와 실행 결과를 연결해 목록에 추가합니다.
            outputs.append({
                # 모델에 돌려줄 항목이 함수 실행 결과임을 명시합니다.
                "type": "function_call_output",
                # 도구 요청과 도구 실행 결과를 연결할 식별자입니다.
                "call_id": call.call_id,
                # Python 실행 결과를 JSON 문자열로 변환해 전달합니다.
                "output": json.dumps(tool_result, ensure_ascii=False),
            # 앞에서 시작한 딕셔너리 항목을 닫습니다.
            })
        # 모델의 출력 또는 다음 도구 호출 요청을 받습니다.
        response = llm_client.responses.create(
            # 사용할 LLM 모델을 지정합니다.
            model=MODEL_NAME,
            # 앞선 응답과 이번 실행 결과를 이어 대화를 계속합니다.
            previous_response_id=response.id,
            # 모델에게 사용자 요청이나 실행 결과를 전달합니다.
            input=outputs,
            # 모델이 선택할 수 있는 도구 목록을 전달합니다.
            tools=TOOLS,
            # 수업 중 결과 추적을 쉽게 하기 위해 병렬 도구 요청을 끕니다.
            parallel_tool_calls=False,
        # 앞에서 시작한 함수 호출 또는 표현식을 닫습니다.
        )
    # 앞선 조건이 성립하지 않은 경우를 처리합니다.
    else:
        # 현재 요청, 실행 결과 또는 상태를 출력하여 확인합니다.
        print("안전을 위해 도구 호출 반복을 4단계에서 중단했습니다.")


선택 실습은 건너뜁니다. 실제 실행하려면 RUN_REAL_LLM=True로 바꾸세요.


## 8. 학습 정리

- **Tool 정의:** 함수 이름·설명·입력 스키마를 모델에 알려줍니다.
- **Tool 요청:** 모델이 함수 이름과 인자(JSON)를 제안합니다. 이때 게임 상태는 아직 바뀌지 않습니다.
- **검증·실행:** 호스트 코드가 스키마·허용 목록·게임 규칙을 확인하고 Python 함수를 실행합니다.
- **결과 반환:** 실행 결과를 다시 모델에 전달하면 모델이 이어서 설명하거나 다른 도구를 요청할 수 있습니다.
- **오류 설계:** `NO_POTION`처럼 오류 종류와 다음 가능한 행동을 알려주면 복구하기 쉽습니다.

**다음 실습(2단계):** 동일한 RPG 함수를 MCP Server의 도구로 공개하고 In-memory Client로 호출합니다.

> 주의: 이번 노트북은 실습용 단순 스키마 검증기를 구현했습니다. 실서비스에서는 완전한 JSON Schema 검증, 권한·승인, 재시도·멱등성 등 별도 정책이 필요합니다.